In [1]:
# pip install transformers datasets evaluate accelerate scikit-learn -U

In [2]:
from datasets import load_dataset
ds = load_dataset("drorrabin/phishing_emails-data")

/home/ltkien/Development/CourseHomework/applied-llm-term-project/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Repo card metadata block was not found. Setting CardData to empty.


### 1. Inspecting the Dataset


In [3]:
# Print the dataset splits and features
print(ds)

# Access the 'train' split and show its features (if available)
if 'train' in ds:
    print("\nFeatures of the 'train' split:")
    print(ds['train'].features)
else:
    print("\nNo 'train' split found. Showing features of the first available split:")
    first_split_name = list(ds.keys())[0]
    print(ds[first_split_name].features)

DatasetDict({
    train: Dataset({
        features: ['text', 'email_type'],
        num_rows: 26946
    })
    test: Dataset({
        features: ['text', 'email_type'],
        num_rows: 3705
    })
})

Features of the 'train' split:
{'text': Value('string'), 'email_type': Value('string')}


### 2. Preprocessing the Data


In [4]:

# Import necessary libraries
from transformers import AutoTokenizer

# Define our label mapping
# We'll inspect the unique labels in 'email_type' to create this mapping properly.
# For now, let's assume 'phishing' and 'not phishing' are the two categories.
# We can refine this after checking unique values if needed.
label_names = ds['train'].unique('email_type')
label_to_id = {label: i for i, label in enumerate(label_names)}
id_to_label = {i: label for i, label in enumerate(label_names)}

print(f"Detected labels: {label_names}")
print(f"Label to ID mapping: {label_to_id}")

# Load a pre-trained tokenizer
# A good starting point for text classification is a BERT-based tokenizer
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Function to tokenize the text and convert labels
def preprocess_function(examples):
    # Tokenize the text
    tokenized_inputs = tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)
    # Convert labels to numerical IDs
    tokenized_inputs["labels"] = [label_to_id[label] for label in examples["email_type"]]
    return tokenized_inputs

# Apply the preprocessing function to the entire dataset
tokenized_ds = ds.map(preprocess_function, batched=True)

# Remove original text and email_type columns, and rename 'labels' column
tokenized_ds = tokenized_ds.remove_columns(["text", "email_type"])

# Set the format for PyTorch, TensorFlow, or JAX
tokenized_ds.set_format("torch")

print("\nTokenized dataset structure:")
print(tokenized_ds)
print("\nExample tokenized item from train split:")
print(tokenized_ds['train'][0])

Detected labels: ['phishing email', 'safe email']
Label to ID mapping: {'phishing email': 0, 'safe email': 1}



Tokenized dataset structure:
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 26946
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 3705
    })
})

Example tokenized item from train split:
{'input_ids': tensor([  101,  2003,  1996,  2206, 10373,  3647,  2030, 13569, 12227,  1029,
         1029,  3058,  1024, 16215,  2226,  1010,  5718, 15476,  2263,  2184,
         1024,  5354,  1024,  3486,  1009,  6185,  8889,  4604,  2121,  1024,
         3679,  2327,  2184,  1026, 10927,  4305,  3567,  2361,  1035,  3261,
         1030,  6627, 23722,  2102,  1012,  4012,  1028,  8393,  1024,  5310,
         2475,  1012,  2260,  1030,  1043, 25465,  1012,  8292,  3022,  1011,
         4119,  1012, 10507, 10373,  3395,  1024, 13229,  1012,  4012,  3679,
         2327,  2184, 10373,  2303,  1024,  1996,  3679,  2327,  2184,  2013,
        13229

### 3. Loading the LLM and Defining Training Arguments

Now we'll load a pre-trained LLM suitable for sequence classification and define the parameters for our fine-tuning process. We'll use `BertForSequenceClassification` from the `transformers` library.

In [5]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
import evaluate

# Load a pre-trained model for sequence classification
# We pass the number of labels and our label mappings
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=len(label_names), id2label=id_to_label, label2id=label_to_id
)

# Define training arguments
training_args = TrainingArguments(
    output_dir="./results",
    # evaluation_strategy="epoch", # Temporarily removed due to TypeError. Re-enable after confirming transformers update.
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    report_to="none" # Disable reporting to services like Weights & Biases
)

# Define metrics for evaluation
metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels, average="macro")

# Create a Trainer instance
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["test"],
    # tokenizer=tokenizer, # Removed this argument as it's not expected in some Trainer versions
    compute_metrics=compute_metrics,
)

print("Model, TrainingArguments, and Trainer configured successfully!")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:  70%|██████▉   | 139/199 [00:00<00:00, 1313.50it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1349.66it/s]


[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model, TrainingArguments, and Trainer configured successfully!


### 4. Training the Model


In [6]:
# Start training
trainer.train()

print("Training complete!")

Step,Training Loss
500,0.061640
1000,0.022107
1500,0.013348
2000,0.009237
2500,0.004296
3000,0.004569
3500,0.002686
4000,0.000024
4500,0.001435
5000,0.000901


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.20it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.20it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.20it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.21it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.21it/s]

Training complete!


In [7]:
trainer.save_model("./my-phishing-classifier")
tokenizer.save_pretrained("./my-phishing-classifier")

print("Model saved to ./my-phishing-classifier")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.16it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.16it/s]

Model saved to ./my-phishing-classifier


### 5. Evaluating the fine-tuned model on the held-out test split

The checkpoint saved by this notebook (`./my-phishing-classifier`, also published as `harrynguyen5/phishing-bert-model`) is loaded and scored on the 3,705-row held-out test split of the **same** dataset. This produces the **in-distribution score**. As §5.1 shows, that score is not a trustworthy measure of phishing-detection ability — it is inflated by a dataset artifact — which is precisely why the team's headline evaluation is performed cross-corpus (80.71% on a disjoint corpus).

In [8]:
# ====================================================
# Step 5: Evaluate on the held-out test split (in-distribution reference)
# ====================================================
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

eval_model = AutoModelForSequenceClassification.from_pretrained("./my-phishing-classifier")

def eval_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    phish_id = label_to_id["phishing email"]
    p, r, f1, _ = precision_recall_fscore_support(
        labels, preds, pos_label=phish_id, average="binary", zero_division=0)
    return {"accuracy": accuracy_score(labels, preds),
            "precision": p, "recall": r, "f1": f1}

eval_trainer = Trainer(
    model=eval_model,
    args=TrainingArguments(output_dir="./eval_tmp", per_device_eval_batch_size=64, report_to="none"),
    compute_metrics=eval_metrics,
)
results = eval_trainer.evaluate(eval_dataset=tokenized_ds["test"])

print("In-distribution performance (held-out test split, N = 3,705):")
for k in ["eval_accuracy", "eval_precision", "eval_recall", "eval_f1"]:
    print(f"  {k[5:]:10}: {results[k]:.4f}")
print("\nCompare with cross-corpus evaluation (notebook 3): accuracy 80.71%, recall 60.43%.")
print("The gap between these two numbers measures the cost of distribution shift.")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 18134.90it/s]

Training Loss,Validation Loss,Step,Accuracy,Precision,Recall,F1
No log,0.000045,0,1.000000,1.000000,1.000000,1.000000


In-distribution performance (held-out test split, N = 3,705):
  accuracy  : 1.0000
  precision : 1.0000
  recall    : 1.0000
  f1        : 1.0000

Compare with cross-corpus evaluation (notebook 3): accuracy 80.71%, recall 60.43%.
The gap between these two numbers measures the cost of distribution shift.


### 5.1 Why the perfect score must NOT be trusted — a dataset artifact

The 100% test accuracy looks like success; it is actually **evidence of a benchmark flaw**. The two classes in this dataset come from **different source corpora**: the marker `ceas-challenge` (a receiver-domain artifact of the CEAS 2008 spam corpus) appears in **99.8% of phishing** emails but only **14.4% of safe** emails. The headers sit at the start of every email — exactly the part that survives `max_length=128` truncation.

A trivial one-line rule — *"if the text contains `ceas-challenge`, predict phishing"* — scores **92.7% on train and 87.2% on test with no machine learning at all**. BERT learns this artifact plus correlated header fingerprints (sender domains, date ranges) and saturates to 100%.

**Consequences:**
- The in-distribution score measures *source-corpus detection*, not phishing detection. It is not a valid quality measure for this checkpoint.
- This fully explains the cross-corpus results (evaluation notebook): on a disjoint corpus with no headers, the fingerprint signal vanishes and accuracy falls to 80.71% (recall 60.4%) — the residual genuine content knowledge.
- **The cross-corpus evaluation is the only honest measurement of this system.** Reporting the in-distribution number without this caveat would be misleading.

In [ ]:
# Reproduce the artifact finding: one string rule, no ML
import pandas as pd
for split in ["train", "test"]:
    _d = pd.DataFrame({"text": ds[split]["text"], "label": ds[split]["email_type"]})
    rule = _d["text"].str.contains("ceas-challenge", case=False, regex=False).map(
        {True: "phishing email", False: "safe email"})
    marker_rate = _d["text"].str.contains("ceas-challenge", case=False, regex=False)
    print(f"{split}: one-rule accuracy = {(rule == _d['label']).mean()*100:.2f}%  | "
          f"marker in phishing: {marker_rate[_d.label=='phishing email'].mean()*100:.1f}%  "
          f"safe: {marker_rate[_d.label=='safe email'].mean()*100:.1f}%")

### 6. Known limitations of this checkpoint (documented for the final report)

Identified during the team's Week 15 evaluation. Kept here so this training notebook is honest about what the published checkpoint is and what a clean re-train should change.

1. **Partial label-suffix leakage.** Every source email ends with the literal line `Email type is: <label>`. This notebook tokenizes the raw text, so for the **~14.5% of training emails that fit within 128 tokens** the answer survives into the training input. A controlled test (evaluation notebook §6.6) shows a content-rich model does not depend on this shortcut, but a clean re-train must strip the prompt prefix and label suffix.
2. **`max_length=128` truncation.** The median training email is ~248 BERT tokens, and the prompt prefix + `Date/Sender/Subject` headers consume a large share of the 128-token budget — for long emails the model saw little or no body content during training. This is the likely root cause of the length degradation found in evaluation (F1 falls from 85% to 41% across length deciles). Re-train at `max_length=512`.
3. **Train/inference format mismatch.** Training inputs carry the prompt prefix and structured headers; the deployed cascade (notebook 2) feeds the cleaned **body only**, at `max_length=512`. The model is deployed on inputs shaped differently from training. The next iteration must use one canonical input format in both places.
4. **Reproducibility.** No random seed is fixed and the label↔id mapping depends on `Dataset.unique()` ordering. Pin `seed=42` and an explicit `label2id = {"phishing email": 0, "safe email": 1}` on re-train.
5. **Single-source class artifact (most important).** The phishing class is sourced almost entirely from the CEAS 2008 corpus (`ceas-challenge` marker: 99.8% of phishing vs 14.4% of safe). Header fingerprints make the classes nearly linearly separable without reading any email content — a one-rule classifier reaches 92.7%. Any in-distribution benchmark on this dataset is therefore unreliable; model quality must be measured cross-corpus (see evaluation notebook).